# Build Phoneme Prototypes with Wav2Vec2-Large

This notebook:
1. Loads **wav2vec2-large-960h** (1024D embeddings instead of 768D)
2. Processes TIMIT training data in batches
3. Saves embeddings incrementally to avoid memory issues
4. Computes centroids and P95 radii
5. Exports PKL + JSON for your app.py

**Expected improvement:** 81.6% → 85-88% accuracy on TIMIT test

## Setup & Configuration

In [1]:
import os

# Limit CPU threads to avoid memory issues
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_MAX_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

print("Thread limits set")

Thread limits set


In [2]:
import torch
import numpy as np
import soundfile as sf
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import pickle
import json
import gc

from transformers import Wav2Vec2Model, Wav2Vec2Processor

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

PyTorch version: 2.9.1
CUDA available: False
MPS available: False


## Configuration

In [3]:
# ==================== CONFIG ====================

# Paths
TIMIT_ROOT = "/Users/samanthajohn/Pronounciation_checker/data_new"
TRAIN_SUBDIR = "TRAIN"
OUTPUT_DIR = "./artifacts"

# THIS IS THE KEY CHANGE: Use LARGE model
MODEL_NAME = "facebook/wav2vec2-large-960h"  # 1024D embeddings

# Audio
SAMPLE_RATE = 16000
MIN_SEG_DUR_MS = 30  # Drop segments shorter than 30ms

# Processing
BATCH_SIZE = 16  # Process 16 segments at once
SAVE_EVERY_N_FILES = 100  # Save progress every 100 files
MAX_FILES = None  # Set to None to process ALL files, or a number to test

# Output files
os.makedirs(OUTPUT_DIR, exist_ok=True)
EMBEDDINGS_NPZ = os.path.join(OUTPUT_DIR, "embeddings_large_train.npz")
PROTOTYPES_PKL = os.path.join(OUTPUT_DIR, "phoneme_prototypes_large.pkl")
PROTOTYPES_JSON = os.path.join(OUTPUT_DIR, "phoneme_prototypes_large.json")

print(f"Model: {MODEL_NAME}")
print(f"Expected embedding dimension: 1024")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Processing {'ALL' if MAX_FILES is None else MAX_FILES} files")

Model: facebook/wav2vec2-large-960h
Expected embedding dimension: 1024
Output directory: ./artifacts
Processing ALL files


## Phone Normalization (39-phone TIMIT set)

In [4]:
# Phones to drop (silence/noise/closures)
DROP_PHONES = {
    "h#", "pau", "epi",
    "bcl", "dcl", "gcl", "kcl", "pcl", "tcl"
}

# Map 61-phone TIMIT to 39-phone set
MAP_61_TO_39 = {
    # Vowels
    "aa":"aa", "ae":"ae", "ah":"ah", "ao":"aa", "aw":"aw", "ax":"ah", "ax-h":"ah", "axr":"er",
    "ay":"ay", "eh":"eh", "el":"l", "em":"m", "en":"n", "eng":"ng", "er":"er",
    "ey":"ey", "ih":"ih", "ix":"ih", "iy":"iy", "ow":"ow", "oy":"oy", "uh":"uh", "uw":"uw", "ux":"uw",
    # Consonants
    "b":"b", "d":"d", "g":"g", "p":"p", "t":"t", "k":"k", "jh":"jh", "ch":"ch",
    "s":"s", "sh":"sh", "z":"z", "zh":"sh", "f":"f", "th":"th", "v":"v", "dh":"dh",
    "m":"m", "n":"n", "ng":"ng", "l":"l", "r":"r", "w":"w", "y":"y", "hh":"hh", "hv":"hh",
    "dx":"dx", "q":"",
}

def normalize_phone(ph: str) -> str:
    """Return normalized 39-phone label or empty string to drop."""
    if ph in DROP_PHONES or ph == "q":
        return ""
    return MAP_61_TO_39.get(ph, ph)

# Test
print("Sample mappings:")
for test_ph in ["aa", "ao", "ix", "zh", "pcl", "h#"]:
    print(f"  {test_ph:5s} → '{normalize_phone(test_ph)}'")

Sample mappings:
  aa    → 'aa'
  ao    → 'aa'
  ix    → 'ih'
  zh    → 'sh'
  pcl   → ''
  h#    → ''


## Load & Index TIMIT Training Data

In [5]:
def load_phn_file(phn_path: Path, wav_path: Path, min_dur_samples: int) -> list:
    """Parse TIMIT .PHN file and return list of (phoneme, start_s, end_s)."""
    segments = []
    
    with open(phn_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue
            
            start_sample, end_sample, phone = parts
            start_sample = int(start_sample)
            end_sample = int(end_sample)
            
            # Normalize phone
            norm_phone = normalize_phone(phone)
            if not norm_phone:
                continue
            
            # Check minimum duration
            if (end_sample - start_sample) < min_dur_samples:
                continue
            
            # Convert to seconds
            start_s = start_sample / SAMPLE_RATE
            end_s = end_sample / SAMPLE_RATE
            
            segments.append((norm_phone, start_s, end_s))
    
    return segments


# Index all training files
train_path = Path(TIMIT_ROOT) / TRAIN_SUBDIR
phn_files = sorted(train_path.rglob("*.PHN"))

print(f"Found {len(phn_files)} .PHN files")

# Group by WAV file with their phoneme segments
min_dur_samples = int(MIN_SEG_DUR_MS / 1000.0 * SAMPLE_RATE)

file_data = []  # List of (wav_path, [(phone, start_s, end_s), ...])
total_segments = 0

for phn_file in tqdm(phn_files, desc="Indexing files"):
    wav_file = phn_file.with_suffix('.WAV')
    if not wav_file.exists():
        continue
    
    segments = load_phn_file(phn_file, wav_file, min_dur_samples)
    if segments:
        file_data.append((str(wav_file), segments))
        total_segments += len(segments)

print(f"\nIndexed {len(file_data)} audio files")
print(f"Total segments (>= {MIN_SEG_DUR_MS}ms): {total_segments}")

# Limit if testing
if MAX_FILES is not None:
    file_data = file_data[:MAX_FILES]
    print(f"Limited to first {len(file_data)} files for testing")

Found 4620 .PHN files


Indexing files:   0%|          | 0/4620 [00:00<?, ?it/s]


Indexed 4620 audio files
Total segments (>= 30ms): 120863


## Initialize Large Wav2Vec2 Model

In [6]:
# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

# Load LARGE model
print(f"Loading {MODEL_NAME}...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f"Model loaded. Embedding dimension: 1024")

Using device: cpu
Loading facebook/wav2vec2-large-960h...


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-large-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded. Embedding dimension: 1024


## Generate Embeddings (Batched, with Checkpointing)

In [7]:
# Storage for embeddings by phoneme
embeddings_by_phone = defaultdict(list)

total_processed = 0
total_skipped = 0

# Progress tracking
pbar = tqdm(total=len(file_data), desc="Processing files")

for file_idx, (wav_path, segments) in enumerate(file_data):
    try:
        # Load audio
        audio, sr = sf.read(wav_path, dtype='float32', always_2d=False)
        
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        
        if sr != SAMPLE_RATE:
            print(f"\nWarning: {Path(wav_path).name} has sr={sr}, skipping")
            pbar.update(1)
            continue
        
        # Process segments in batches
        for batch_start in range(0, len(segments), BATCH_SIZE):
            batch_segments = segments[batch_start:batch_start + BATCH_SIZE]
            
            batch_waves = []
            batch_phones = []
            
            for phone, start_s, end_s in batch_segments:
                start_idx = int(start_s * sr)
                end_idx = int(end_s * sr)
                
                # Bounds check
                if end_idx <= start_idx or start_idx < 0 or end_idx > len(audio):
                    total_skipped += 1
                    continue
                
                seg_audio = audio[start_idx:end_idx]
                
                if len(seg_audio) < 160:  # ~10ms minimum
                    total_skipped += 1
                    continue
                
                batch_waves.append(seg_audio.astype('float32'))
                batch_phones.append(phone)
            
            if not batch_waves:
                continue
            
            # Batch process with model
            with torch.no_grad():
                inputs = processor(
                    batch_waves,
                    sampling_rate=SAMPLE_RATE,
                    return_tensors="pt",
                    padding=True
                )
                
                outputs = model(
                    inputs.input_values.to(device),
                    output_hidden_states=True
                )
                
                # Mean pool over time: (B, T, 1024) -> (B, 1024)
                batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            
            # Store by phoneme
            for phone, emb in zip(batch_phones, batch_embeddings):
                embeddings_by_phone[phone].append(emb.astype('float32'))
                total_processed += 1
        
        # Checkpoint every N files
        if (file_idx + 1) % SAVE_EVERY_N_FILES == 0:
            checkpoint_path = os.path.join(OUTPUT_DIR, f"checkpoint_{file_idx+1}.npz")
            np.savez_compressed(
                checkpoint_path,
                **{ph: np.array(embs) for ph, embs in embeddings_by_phone.items()}
            )
            print(f"\n[Checkpoint] Saved at file {file_idx+1}/{len(file_data)}")
            print(f"  Phonemes: {len(embeddings_by_phone)}")
            print(f"  Total embeddings: {sum(len(v) for v in embeddings_by_phone.values())}")
        
    except Exception as e:
        print(f"\nError processing {Path(wav_path).name}: {e}")
    
    pbar.update(1)

pbar.close()

print(f"\n{'='*60}")
print(f"EMBEDDING COMPLETE")
print(f"{'='*60}")
print(f"Total segments processed: {total_processed}")
print(f"Total segments skipped: {total_skipped}")
print(f"Unique phonemes: {len(embeddings_by_phone)}")
print(f"Total embeddings: {sum(len(v) for v in embeddings_by_phone.values())}")

# Print per-phoneme counts
print(f"\nPer-phoneme counts:")
for phone in sorted(embeddings_by_phone.keys()):
    print(f"  {phone:3s}: {len(embeddings_by_phone[phone]):5d} samples")

Processing files:   0%|          | 0/4620 [00:00<?, ?it/s]


[Checkpoint] Saved at file 100/4620
  Phonemes: 39
  Total embeddings: 2566

[Checkpoint] Saved at file 200/4620
  Phonemes: 39
  Total embeddings: 5127

[Checkpoint] Saved at file 300/4620
  Phonemes: 39
  Total embeddings: 7688

[Checkpoint] Saved at file 400/4620
  Phonemes: 39
  Total embeddings: 10282

[Checkpoint] Saved at file 500/4620
  Phonemes: 39
  Total embeddings: 12889

[Checkpoint] Saved at file 600/4620
  Phonemes: 39
  Total embeddings: 15510

[Checkpoint] Saved at file 700/4620
  Phonemes: 39
  Total embeddings: 18069

[Checkpoint] Saved at file 800/4620
  Phonemes: 39
  Total embeddings: 20718

[Checkpoint] Saved at file 900/4620
  Phonemes: 39
  Total embeddings: 23323

[Checkpoint] Saved at file 1000/4620
  Phonemes: 39
  Total embeddings: 25943

[Checkpoint] Saved at file 1100/4620
  Phonemes: 39
  Total embeddings: 28571

[Checkpoint] Saved at file 1200/4620
  Phonemes: 39
  Total embeddings: 31209

[Checkpoint] Saved at file 1300/4620
  Phonemes: 39
  Total emb

## Save Final Embeddings

In [8]:
print(f"Saving embeddings to {EMBEDDINGS_NPZ}...")

np.savez_compressed(
    EMBEDDINGS_NPZ,
    **{phone: np.array(embs) for phone, embs in embeddings_by_phone.items()}
)

print(f"✓ Saved {len(embeddings_by_phone)} phonemes")
print(f"  File size: {os.path.getsize(EMBEDDINGS_NPZ) / 1024 / 1024:.1f} MB")

Saving embeddings to ./artifacts/embeddings_large_train.npz...
✓ Saved 39 phonemes
  File size: 439.4 MB


## Compute Prototypes (Centroids + P95 Radii)

In [9]:
print("Computing prototypes...\n")

prototypes = {}
p95_data = {}

for phone, emb_list in sorted(embeddings_by_phone.items()):
    embs = np.array(emb_list)  # Shape: (N, 1024)
    
    if len(embs) < 10:
        print(f"Warning: {phone} has only {len(embs)} samples, skipping")
        continue
    
    # Compute centroid
    centroid = np.mean(embs, axis=0)
    
    # Compute distances from centroid
    dists = np.linalg.norm(embs - centroid, axis=1)
    
    # Statistics
    mean_dist = np.mean(dists)
    std_dist = np.std(dists)
    p95_radius = np.percentile(dists, 95)
    
    # Store in PKL format (for runtime)
    prototypes[phone] = {
        'centroid': centroid.astype('float32'),
        'radius': p95_radius,  # Use P95 as the radius
        'count': len(embs)
    }
    
    # Store in JSON format (for reference)
    p95_data[phone] = {
        'n': int(len(embs)),
        'mean_dist': float(mean_dist),
        'std_dist': float(std_dist),
        'p95_radius': float(p95_radius)
    }
    
    print(f"{phone:3s}: n={len(embs):5d}, mean_dist={mean_dist:.3f}, p95={p95_radius:.3f}")

print(f"\n{'='*60}")
print(f"PROTOTYPES COMPUTED")
print(f"{'='*60}")
print(f"Total phonemes: {len(prototypes)}")

Computing prototypes...

aa : n= 5999, mean_dist=4.997, p95=8.014
ae : n= 3997, mean_dist=4.855, p95=7.714
ah : n= 5632, mean_dist=4.603, p95=8.042
aw : n=  729, mean_dist=4.964, p95=7.818
ay : n= 2390, mean_dist=5.131, p95=8.204
b  : n=  134, mean_dist=6.120, p95=10.130
ch : n=  821, mean_dist=5.301, p95=8.346
d  : n=  759, mean_dist=5.312, p95=8.532
dh : n= 1628, mean_dist=5.658, p95=8.356
dx : n= 1195, mean_dist=5.415, p95=8.619
eh : n= 3842, mean_dist=4.491, p95=7.449
er : n= 5405, mean_dist=4.584, p95=7.604
ey : n= 2282, mean_dist=5.096, p95=7.869
f  : n= 2193, mean_dist=6.145, p95=8.969
g  : n=  890, mean_dist=5.504, p95=8.596
hh : n= 2005, mean_dist=5.462, p95=8.430
ih : n=12808, mean_dist=4.167, p95=7.160
iy : n= 6941, mean_dist=5.041, p95=7.825
jh : n= 1137, mean_dist=5.496, p95=8.238
k  : n= 3651, mean_dist=5.499, p95=8.327
l  : n= 6489, mean_dist=5.382, p95=8.453
m  : n= 3660, mean_dist=5.813, p95=8.743
n  : n= 7094, mean_dist=5.276, p95=8.157
ng : n= 1302, mean_dist=5.496, 

## Save Prototypes (PKL + JSON)

In [10]:
# Save PKL (for runtime use)
print(f"Saving prototypes to {PROTOTYPES_PKL}...")
with open(PROTOTYPES_PKL, 'wb') as f:
    pickle.dump(prototypes, f)
print(f"✓ Saved PKL ({os.path.getsize(PROTOTYPES_PKL) / 1024:.1f} KB)")

# Save JSON (human-readable reference)
print(f"Saving statistics to {PROTOTYPES_JSON}...")
with open(PROTOTYPES_JSON, 'w') as f:
    json.dump(p95_data, f, indent=2)
print(f"✓ Saved JSON ({os.path.getsize(PROTOTYPES_JSON) / 1024:.1f} KB)")

print(f"\n{'='*60}")
print(f"ALL DONE!")
print(f"{'='*60}")
print(f"\nGenerated files:")
print(f"  1. {EMBEDDINGS_NPZ}")
print(f"  2. {PROTOTYPES_PKL}")
print(f"  3. {PROTOTYPES_JSON}")
print(f"\nNext steps:")
print(f"  1. Update app.py to use large model and new prototypes")
print(f"  2. Update models_runtime/embedding_runtime.py to use wav2vec2-large")
print(f"  3. Re-run evaluation: python batch_evaluate_timit.py")
print(f"\nExpected accuracy improvement: 81.6% → 85-88%")

Saving prototypes to ./artifacts/phoneme_prototypes_large.pkl...
✓ Saved PKL (158.7 KB)
Saving statistics to ./artifacts/phoneme_prototypes_large.json...
✓ Saved JSON (5.2 KB)

ALL DONE!

Generated files:
  1. ./artifacts/embeddings_large_train.npz
  2. ./artifacts/phoneme_prototypes_large.pkl
  3. ./artifacts/phoneme_prototypes_large.json

Next steps:
  1. Update app.py to use large model and new prototypes
  2. Update models_runtime/embedding_runtime.py to use wav2vec2-large
  3. Re-run evaluation: python batch_evaluate_timit.py

Expected accuracy improvement: 81.6% → 85-88%


### Testing

In [11]:
"""
Recompute prototypes with proper normalization for large model.
The issue: Large model embeddings have larger magnitudes, so raw P95 is too loose.
Solution: Normalize embeddings before computing prototypes.
"""

import numpy as np
import pickle
import json
from sklearn.preprocessing import normalize

print("Loading embeddings from artifacts/embeddings_large_train.npz...")
data = np.load("artifacts/embeddings_large_train.npz")

prototypes = {}
p95_data = {}

print("\nComputing prototypes with L2 normalization...\n")

for phone in sorted(data.files):
    embs = data[phone]  # Shape: (N, 1024)
    
    if len(embs) < 10:
        print(f"Skipping {phone}: only {len(embs)} samples")
        continue
    
    # CRITICAL: L2-normalize embeddings to unit sphere
    # This makes distances more stable across different model sizes
    embs_norm = normalize(embs, norm='l2', axis=1)
    
    # Compute centroid on normalized embeddings
    centroid = np.mean(embs_norm, axis=0)
    
    # Re-normalize centroid (it might drift slightly off unit sphere)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
    
    # Compute distances from centroid
    dists = np.linalg.norm(embs_norm - centroid, axis=1)
    
    # Statistics
    mean_dist = np.mean(dists)
    std_dist = np.std(dists)
    p95_radius = np.percentile(dists, 95)
    p90_radius = np.percentile(dists, 90)
    
    # Store in PKL format (for runtime)
    prototypes[phone] = {
        'centroid': centroid.astype('float32'),
        'radius': p95_radius,
        'count': len(embs)
    }
    
    # Store in JSON format (for reference)
    p95_data[phone] = {
        'n': int(len(embs)),
        'mean_dist': float(mean_dist),
        'std_dist': float(std_dist),
        'p90_radius': float(p90_radius),
        'p95_radius': float(p95_radius)
    }
    
    print(f"{phone:3s}: n={len(embs):5d}, mean={mean_dist:.4f}, "
          f"p90={p90_radius:.4f}, p95={p95_radius:.4f}")

print(f"\n{'='*60}")
print(f"PROTOTYPES COMPUTED")
print(f"{'='*60}")
print(f"Total phonemes: {len(prototypes)}")

# Save PKL
pkl_path = "artifacts/phoneme_prototypes_large.pkl"
print(f"\nSaving to {pkl_path}...")
with open(pkl_path, 'wb') as f:
    pickle.dump(prototypes, f)
print(f"✓ Saved ({len(prototypes)} phonemes)")

# Save JSON
json_path = "artifacts/phoneme_prototypes_large.json"
print(f"Saving to {json_path}...")
with open(json_path, 'w') as f:
    json.dump(p95_data, f, indent=2)
print(f"✓ Saved")

# Print expected radius ranges
all_p95 = [d['p95_radius'] for d in p95_data.values()]
print(f"\nP95 Radius Statistics:")
print(f"  Min:    {min(all_p95):.4f}")
print(f"  Median: {np.median(all_p95):.4f}")
print(f"  Max:    {max(all_p95):.4f}")
print(f"  Mean:   {np.mean(all_p95):.4f}")

print(f"\nExpected distances after normalization:")
print(f"  Should be in range [0.3, 0.6] for most segments")
print(f"  Radii should be in range [0.4, 0.8]")

Loading embeddings from artifacts/embeddings_large_train.npz...

Computing prototypes with L2 normalization...

aa : n= 5999, mean=0.4224, p90=0.5688, p95=0.6419
ae : n= 3997, mean=0.4155, p90=0.5627, p95=0.6383
ah : n= 5632, mean=0.3849, p90=0.5605, p95=0.6340
aw : n=  729, mean=0.4271, p90=0.5887, p95=0.6650
ay : n= 2390, mean=0.4439, p90=0.6050, p95=0.6847
b  : n=  134, mean=0.4431, p90=0.6178, p95=0.7298
ch : n=  821, mean=0.4098, p90=0.5375, p95=0.6081
d  : n=  759, mean=0.3935, p90=0.5654, p95=0.6273
dh : n= 1628, mean=0.4231, p90=0.5622, p95=0.6219
dx : n= 1195, mean=0.4056, p90=0.5657, p95=0.6423
eh : n= 3842, mean=0.3822, p90=0.5375, p95=0.6093
er : n= 5405, mean=0.3865, p90=0.5422, p95=0.6187
ey : n= 2282, mean=0.4250, p90=0.5641, p95=0.6260
f  : n= 2193, mean=0.4614, p90=0.5970, p95=0.6602
g  : n=  890, mean=0.4027, p90=0.5613, p95=0.6301
hh : n= 2005, mean=0.4274, p90=0.5683, p95=0.6203
ih : n=12808, mean=0.3492, p90=0.5076, p95=0.5769
iy : n= 6941, mean=0.4138, p90=0.5513,